In [4]:
import fitz  # PyMuPDF
from transformers import pipeline
from typing import List

# 🔹 Chargement du modèle francophone
qa = pipeline(
    "question-answering",
    model="etalab-ia/camembert-base-squadFR-fquad-piaf",
    tokenizer="etalab-ia/camembert-base-squadFR-fquad-piaf",
    use_fast=False
)

# 🔹 Lire le PDF et découper en pages (chunks)
def extraire_chunks(pdf_path: str) -> List[str]:
    doc = fitz.open(pdf_path)
    return [page.get_text() for page in doc if page.get_text().strip()]

# 🔹 Poser une question sur tous les chunks
def repondre(question: str, chunks: List[str]) -> str:
    meilleures_reponses = []

    for chunk in chunks:
        try:
            reponse = qa(question=question, context=chunk)
            meilleures_reponses.append(reponse)
        except Exception:
            continue  # sauter les erreurs éventuelles

    if not meilleures_reponses:
        return "Aucune réponse trouvée."

    # Garder la réponse avec le meilleur score
    meilleures_reponses.sort(key=lambda x: x['score'], reverse=True)
    return meilleures_reponses[0]['answer']

# 🔹 Interface principale
def agent_pdf(pdf_path: str):
    print("📄 Lecture du document PDF...")
    chunks = extraire_chunks(pdf_path)
    print(f"✅ Document chargé ({len(chunks)} morceaux)")

    while True:
        question = input("\n❓ Pose ta question (ou tape 'exit' pour quitter):\n> ")
        if question.lower() in {"exit", "quit"}:
            break
        reponse = repondre(question, chunks)
        print("📌 Réponse :", reponse)

# 🔹 Lancer l'agent
if __name__ == "__main__":
    chemin_pdf = "test"  # ← Remplace par le nom de ton PDF
    agent_pdf(chemin_pdf)


Device set to use mps:0


📄 Lecture du document PDF...


FileNotFoundError: no such file: 'test'